[← 08b - Changing Data, Part 2](<08b-changing-data-part-2.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# 09 - Thinking in Metrics: one business question, four correct queries

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/09-thinking-in-metrics.ipynb)

`08b` ended with a question that has nothing to do with syntax: you can now read and write every row in these tables, so how does a business question like "who are our best customers?" turn into a query? Take that question literally and hand it to the same small dataset you've used since `04`, and SQL can answer it at least four ways, every one of them correct, and each way puts a different customer at the top. This notebook is about the part SQL can't do for you, which is deciding what to ask.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** turn a business question into a written metric definition before writing SQL, spot the decisions a question leaves open (grain, measure, who counts, period, and ties), and catch the two traps that silently change a metric: dividing two whole numbers, and `LIMIT 1` sitting on a tie.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 9 of the All About Data & More curriculum](LESSON_9_URL), which walks through the reasoning in more depth.

## 0. Set up

Run the cell below once each time you open the notebook. After it, any cell that starts with `%%sql` runs SQLite against a small lesson database that lives in memory, so there's nothing to install, and any cell that starts with `%%tsql` runs against SQL Server and AdventureWorks, which only section 9 uses and which has its own one-step setup there.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. The same customers and orders

Everything in this notebook only reads from the tables. If you've just worked through `08a` or `08b` in the same session, run `%sql_reset` first, so the rows match `04` to `07` exactly.

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


## 2. One question, four correct answers

"Best" could reasonably mean spent the most, ordered most often, placed the biggest orders, or used the most of their credit. Here are all four, side by side, built on one CTE of the kind `05` introduced:

In [3]:
%%sql
WITH customer_stats AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.credit_limit,
        COALESCE(SUM(o.amount), 0) AS total_spent,
        COUNT(o.order_id) AS order_count,
        AVG(o.amount) AS avg_order
    FROM customers AS c
    LEFT JOIN orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.credit_limit
)
SELECT
    customer_name,
    total_spent,
    order_count,
    avg_order,
    ROUND(100.0 * total_spent / credit_limit, 1) AS pct_of_limit
FROM customer_stats
ORDER BY
    customer_id;

customer_name,total_spent,order_count,avg_order,pct_of_limit
Ada,100,2,50.0,100.0
Bola,170,2,85.0,85.0
Chidi,140,1,140.0,46.7
Dami,0,0,NULL,0.0


Read it one column at a time. By total spent, Bola is the best customer. By order count, Ada and Bola tie. By average order size, Chidi wins comfortably, with one order bigger than anything Ada or Bola has placed. By share of credit limit used, Ada is at 100 percent and nobody else is close. Dami comes last everywhere, and her `avg_order` is `NULL` rather than zero, because, as `03` showed, `AVG()` over no amounts at all has nothing to average.

<p align="center">
  <img src="graphics/09_four_definitions.png" width="640" alt="Who are our best customers? By total spent, Bola with 170; by number of orders, Ada and Bola tie with 2; by average order size, Chidi with 140; by share of credit limit used, Ada at 100 percent">
</p>

Nothing in that query is wrong. Each column answers its own question correctly, and the original question never said which one it meant. (The `100.0` instead of `100` in the last column is deliberate, and section 5 shows what goes wrong without it.)

## 3. A metric is a definition before it's a query

A **metric** is a number with a definition attached: a precise statement of what gets measured, over which rows, for what period. "Best customers" isn't a metric yet; it's a question with several decisions still open, and a query can't run until every one of them is settled, whether you settled it on purpose or not.

<p align="center">
  <img src="graphics/09_question_to_query.png" width="860" alt="The business question passes through five decisions, grain, measure, who counts, period, and ties, before becoming a definition written as a sentence and then the query that implements it">
</p>

### What one row of the answer is

This is the grain from `04`, applied to the answer instead of the table. "Best customers" wants one row per customer, so the orders have to be collapsed to one row per customer before anything gets ranked. Rank the `orders` table directly and you've answered a different question, "what's our best order?":

In [4]:
%%sql
SELECT
    order_id,
    customer_id,
    amount
FROM orders
ORDER BY
    amount DESC
LIMIT 1;

order_id,customer_id,amount
105,3,140


Chidi's order 105, which says nothing about whether Chidi is the best *customer*.

### What gets measured

That's the choice between the four columns in section 2, and the data can't make it for you. The right one depends on what the answer will be used for: if the plan is to thank the people who brought in the most revenue, `total_spent` fits, while a team reviewing who's close to their credit limit needs `pct_of_limit`. You find the measure by asking what decision the number will feed.

### Who counts

Dami has never ordered. Is she one of the customers being compared, or not a customer for this purpose at all? It sounds like a small question, and it changes averages, as section 4 shows.

### Over what period

"Best customers this quarter" is a common way to ask this question, and these tables can't answer it, because `orders` has no date column. When that happens, the honest move is to say so, not to quietly answer for all time and let the reader assume otherwise. (AdventureWorks does store order dates, and the stretch exercise shows the period changing the answer.)

### What happens on a tie

Ada and Bola both have two orders. If the answer is supposed to name one customer, something has to break the tie, and if nothing does, SQL will pick one for you without saying so, which is section 6.

## 4. Edge case: who counts in "average spend per customer"

`05` compared each customer to "the average among customers who've placed at least one order", and that average was 136.67; `06` rebuilt the same comparison and got the same number. Both said who counted, which is why the number was right. But ask for "average spend per customer" without that phrase, and Dami is a customer too.

You might expect that a `LEFT JOIN`, which keeps Dami, would be enough to include her. It isn't:

In [5]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        SUM(o.amount) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY
        c.customer_id
)
SELECT
    ROUND(AVG(total_spent), 2) AS avg_without_coalesce,
    ROUND(AVG(COALESCE(total_spent, 0)), 2) AS avg_with_coalesce
FROM customer_totals;

avg_without_coalesce,avg_with_coalesce
136.67,102.5


The `LEFT JOIN` did keep Dami's row, but `SUM()` over a group with no amounts returns `NULL`, not zero, and the [SQLite aggregate function documentation](https://www.sqlite.org/lang_aggfunc.html) says `AVG()` "returns the average value of all non-NULL X within a group." So her row reaches the average and is skipped anyway, and the first column quietly gives you the "customers who ordered" figure from `05`. Only `COALESCE`, which turns her missing total into a real zero, gets her counted: 410 divided by 4, which is 102.5.

<p align="center">
  <img src="graphics/09_average_spend.png" width="520" alt="INNER JOIN gives 136.67 for ordering customers only; LEFT JOIN without COALESCE keeps Dami but AVG skips her NULL, giving 136.67 again; LEFT JOIN plus COALESCE gives 102.5 across all four; whole-number division of 410 by 4 gives 102">
</p>

Neither 136.67 nor 102.5 is the wrong answer. They answer two different questions, and the mistake would be reporting either one under a label that doesn't say which. A column named `avg_spend_per_ordering_customer` is clumsy, and it's also impossible to misread.

## 5. Common mistake: dividing two whole numbers

Suppose you write the share of credit limit the obvious way, without the `100.0`:

In [6]:
%%sql
SELECT
    c.customer_name,
    SUM(o.amount) / c.credit_limit AS share_of_limit
FROM customers AS c
INNER JOIN orders AS o
    ON o.customer_id = c.customer_id
GROUP BY
    c.customer_id,
    c.customer_name,
    c.credit_limit
ORDER BY
    c.customer_id;

customer_name,share_of_limit
Ada,1
Bola,0
Chidi,0


Bola has used 170 of her 200, which is 0.85, and the query says 0. Both columns hold whole numbers, and the [SQLite expression documentation](https://www.sqlite.org/lang_expr.html) says what happens then: "Integer divide yields an integer result, truncated toward zero." So 170 divided by 200 becomes 0, 140 divided by 300 becomes 0, and Ada's exact 100 out of 100 is the only value that survives. Nothing errors, and a report built on this would say two of your three ordering customers have used none of their credit.

The fix is to make one side of the division a decimal before it happens, by multiplying by `1.0` (or `100.0` for a percentage), which is also why `05` wrote `credit_limit / 2.0` and not `credit_limit / 2`:

In [7]:
%%sql
SELECT
    c.customer_name,
    ROUND(SUM(o.amount) * 1.0 / c.credit_limit, 2) AS share_of_limit
FROM customers AS c
INNER JOIN orders AS o
    ON o.customer_id = c.customer_id
GROUP BY
    c.customer_id,
    c.customer_name,
    c.credit_limit
ORDER BY
    c.customer_id;

customer_name,share_of_limit
Ada,1.0
Bola,0.85
Chidi,0.47


The same trap catches a hand-built average, which is worth seeing because it looks so reasonable:

In [8]:
%%sql
SELECT
    (SELECT SUM(amount) FROM orders) / (SELECT COUNT(*) FROM customers) AS hand_built_average,
    (SELECT SUM(amount) FROM orders) * 1.0 / (SELECT COUNT(*) FROM customers) AS with_decimal;

hand_built_average,with_decimal
102,102.5


102, not 102.5. SQLite's `AVG()` doesn't have this problem, because its result "is always a floating point value" whenever there's at least one non-`NULL` input.

This is also an engine difference worth knowing before you change jobs or tools. PostgreSQL's [documentation](https://www.postgresql.org/docs/current/functions-math.html) says division of integer types "truncates the result towards zero", the same as SQLite, and SQL Server does the same, as section 9 shows. MySQL's `/` operator returns a decimal instead, and its [documentation](https://dev.mysql.com/doc/refman/8.4/en/arithmetic-functions.html) shows `3/5` returning 0.60. The same query can report 0 on one engine and 0.85 on another, so write the `1.0` and the result stops depending on where it runs.

## 6. Common mistake: LIMIT 1 when there's a tie

"Who's our most frequent customer?" looks like it wants one name, so the natural query sorts and keeps the top row:

In [9]:
%%sql
SELECT
    c.customer_name,
    COUNT(o.order_id) AS order_count
FROM customers AS c
LEFT JOIN orders AS o
    ON o.customer_id = c.customer_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    order_count DESC
LIMIT 1;

customer_name,order_count
Ada,2


Ada, with 2. Bola also has 2, and she's gone. `07` quoted the rule that applies: when two rows have equal `ORDER BY` values, their order is undefined, so `LIMIT 1` keeps whichever one happens to come first, and a different engine, or the same engine on a bigger table, is free to pick Bola instead. The query gives no sign that a second customer had an equal claim.

If the honest answer is "these two, jointly", use `RANK()`, a sibling of `06`'s `ROW_NUMBER()` that gives tied rows the same number. The [SQLite window function documentation](https://www.sqlite.org/windowfunctions.html) describes it as "the rank of the current row with gaps", so two customers tied for first are both rank 1 and the next one is rank 3. Since a window function's alias can't be filtered in its own `WHERE`, the ranking goes in a CTE first:

In [10]:
%%sql
WITH customer_orders AS (
    SELECT
        c.customer_name,
        COUNT(o.order_id) AS order_count
    FROM customers AS c
    LEFT JOIN orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
),
ranked AS (
    SELECT
        customer_name,
        order_count,
        RANK() OVER (ORDER BY order_count DESC) AS order_rank
    FROM customer_orders
)
SELECT
    customer_name,
    order_count
FROM ranked
WHERE order_rank = 1;

customer_name,order_count
Ada,2
Bola,2


Both Ada and Bola, each with 2. If the business really does need exactly one name, the tie needs a rule, say more orders first and then higher total spend, and that rule belongs in the `ORDER BY` so the tie stops existing, the same fix `07` used. What doesn't belong in a report is a tiebreaker nobody chose.

## 7. Write the definition down first

Every trap in this notebook was the same trap wearing different clothes: a decision the question didn't make, made silently by the query. The habit that catches all of them is to write the definition as a sentence before writing any SQL, and to keep that sentence with the query, where the next person to open it will see it. For "best customers", one defensible version looks like this:

In [11]:
%%sql
-- Best customers: total order amount, all time (orders has no date column),
-- every customer included (no orders counts as 0), tied totals share a rank.
WITH customer_totals AS (
    SELECT
        c.customer_name,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
)
SELECT
    customer_name,
    total_spent,
    RANK() OVER (ORDER BY total_spent DESC) AS spend_rank
FROM customer_totals
ORDER BY
    spend_rank;

customer_name,total_spent,spend_rank
Bola,170,1
Chidi,140,2
Ada,100,3
Dami,0,4


Each phrase in that comment answers one of the decisions in section 3, so someone who disagrees with the result can point at a phrase instead of at the whole query.

## 8. Try it yourself

Write each query in a new cell before looking at the [solutions notebook](<solutions/09-thinking-in-metrics-solutions.ipynb>), predicting the result first.

1. **Two kinds of average order value.** Average order value is a familiar number on sales dashboards, and it hides the same kind of choice. Before running them, predict whether these two return the same number, and if not, which is bigger:
   ```sql
   SELECT AVG(amount) AS avg_order_value
   FROM orders;
   ```
   ```sql
   WITH customer_averages AS (
       SELECT
           customer_id,
           AVG(amount) AS customer_avg
       FROM orders
       GROUP BY
           customer_id
   )
   SELECT ROUND(AVG(customer_avg), 2) AS avg_of_customer_averages
   FROM customer_averages;
   ```
2. **RANK vs DENSE_RANK.** Rank every customer by order count with both `RANK()` and `DENSE_RANK()`. Where do the two columns differ, and when would you want each one?
3. **Share of all revenue.** For every customer, including Dami, show their share of all revenue as a percentage with one decimal place. The four shares should add up to 100.
4. **Stretch, on AdventureWorks.** Who was the best customer by total due in 2013 alone, and is that the same customer who tops the all-time list? Save this one for after section 9.

## 9. Scale it up: definitions on real data

Four customers make every definition visible. Real data is where two teams pick different definitions without noticing and present different "best customers" at the same meeting. This section runs the lesson's traps against **AdventureWorks2022**, Microsoft's sample database for an invented bike company, on SQL Server.

### Set up AdventureWorks

SQL Server is a separate program from this notebook, so it needs one setup step before the `%%tsql` cells can run:

- **In Google Colab**, the cell below installs SQL Server and AdventureWorks inside your Colab session for you. The first run takes a few minutes and prints each step as it goes. Colab gives you a fresh machine every session, so run it again whenever you come back to the notebook later.
- **On your own machine**, it checks that SQL Server is reachable and tells you what's missing if it isn't; the [Course Overview](<00-course-overview.ipynb>) covers installing SQL Server and AdventureWorks locally.

If you'd rather not set anything up, every `%%tsql` cell below keeps its saved result, so you can read the section without running it.

In [12]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


### Three definitions, three different winners

One query ranks every customer three ways with `RANK()`, then keeps anyone who's first by any of them:

In [13]:
%%tsql
WITH customer_stats AS (
    SELECT
        CustomerID,
        SUM(TotalDue) AS total_spent,
        COUNT(*) AS orders,
        AVG(TotalDue) AS avg_order
    FROM Sales.SalesOrderHeader
    GROUP BY
        CustomerID
),
ranked AS (
    SELECT
        CustomerID,
        total_spent,
        orders,
        avg_order,
        RANK() OVER (ORDER BY total_spent DESC) AS rank_by_total,
        RANK() OVER (ORDER BY orders DESC) AS rank_by_orders,
        RANK() OVER (ORDER BY avg_order DESC) AS rank_by_avg_order
    FROM customer_stats
)
SELECT
    CustomerID,
    total_spent,
    orders,
    avg_order,
    rank_by_total,
    rank_by_orders,
    rank_by_avg_order
FROM ranked
WHERE rank_by_total = 1
   OR rank_by_orders = 1
   OR rank_by_avg_order = 1
ORDER BY
    CustomerID;

CustomerID,total_spent,orders,avg_order,rank_by_total,rank_by_orders,rank_by_avg_order
11091,1314.2103,28,46.9360,7742,1,14253
11176,1458.6118,28,52.0932,7506,1,13967
29641,606819.6087,4,151704.9021,27,408,1
29818,989184.0820,12,82432.0068,1,36,21


Four customers, and no one is first on more than one measure. Customer 29818 spent the most; customer 29641 places the biggest orders on average; and two customers, 11091 and 11176, tie for the most orders, with 28 each. A `TOP 1` on order count would have named one of them and silently dropped the other, which is section 6 on real data.

Notice the shape of the winners too. The two big spenders are stores, placing a handful of very large orders each (customer 29641 placed four), the kind a shop buying bikes for resale would place, while the two most frequent customers are individuals ordering online, 28 small orders worth roughly $1,300 to $1,500 in total. "Best" isn't only a choice of column; it decides which *kind* of customer the business pays attention to.

### Integer division at scale

What share of orders were placed online? `OnlineOrderFlag` is 1 for an online order, so counting the 1s and dividing by all orders looks right:

In [14]:
%%tsql
SELECT
    SUM(CASE WHEN OnlineOrderFlag = 1 THEN 1 ELSE 0 END) / COUNT(*) AS share_whole_numbers,
    SUM(CASE WHEN OnlineOrderFlag = 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS share_with_decimal
FROM Sales.SalesOrderHeader;

share_whole_numbers,share_with_decimal
0,0.879040203400


The first column says no orders were placed online. The second says 88 percent were. SQL Server divides two whole numbers as whole numbers, exactly as SQLite does, and the online share became 0 without a word.

### Who counts, at scale

"Average spend per customer", with and without the customers who never ordered:

In [15]:
%%tsql
WITH customer_totals AS (
    SELECT
        c.CustomerID,
        SUM(h.TotalDue) AS total_spent
    FROM Sales.Customer AS c
    LEFT JOIN Sales.SalesOrderHeader AS h
        ON h.CustomerID = c.CustomerID
    GROUP BY
        c.CustomerID
)
SELECT
    COUNT(*) AS customers,
    COUNT(total_spent) AS customers_who_ordered,
    AVG(total_spent) AS avg_per_ordering_customer,
    AVG(COALESCE(total_spent, 0)) AS avg_per_customer
FROM customer_totals;

customers,customers_who_ordered,avg_per_ordering_customer,avg_per_customer
19820,19119,6444.7296,6216.7904


The 701 customers who never ordered move the average by more than $200. Both numbers are correct; only a label tells the reader which one they're looking at.

### Two kinds of average order value

Exercise 1 asks you to predict these on the lesson data. Here they are on real data:

In [16]:
%%tsql
WITH customer_averages AS (
    SELECT
        CustomerID,
        AVG(TotalDue) AS customer_avg
    FROM Sales.SalesOrderHeader
    GROUP BY
        CustomerID
)
SELECT
    (SELECT AVG(TotalDue) FROM Sales.SalesOrderHeader) AS avg_order_value,
    AVG(customer_avg) AS avg_of_customer_averages
FROM customer_averages;

avg_order_value,avg_of_customer_averages
3915.9951,1678.1178


About $3,916 per order, but only about $1,678 for the typical customer's typical order, so the second number is *smaller* here, the opposite direction from the lesson data. Thousands of small online customers each get one equal vote in the second average, while the 635 store customers, placing far larger orders, carry much more weight in the first. Which way the gap goes depends on the data; that there's a gap at all is the reason to write the definition down.

## Before you trust a metric

- Write the definition as a sentence first: what one row is, what's measured, who counts, over what period, and what happens on a tie.
- Keep that sentence with the query, as a comment.
- Multiply by `1.0` before dividing, so the answer doesn't depend on the engine.
- Use `RANK()` instead of `LIMIT 1` or `TOP 1` when a tie for first is possible.
- Name columns for what they measure (`avg_spend_per_ordering_customer`), even when it's clumsy.
- When two teams report different numbers for the same question, compare their definitions before their SQL.

## Go deeper

- **[Aggregate functions](https://www.sqlite.org/lang_aggfunc.html)** and **[Window functions](https://www.sqlite.org/windowfunctions.html)**, SQLite. How `AVG()` treats `NULL`, and the exact definitions of `RANK()` and `DENSE_RANK()`.
- **[Mathematical functions and operators](https://www.postgresql.org/docs/current/functions-math.html)**, PostgreSQL, and **[Arithmetic operators](https://dev.mysql.com/doc/refman/8.4/en/arithmetic-functions.html)**, MySQL. The two engines on either side of the integer-division difference.
- **[RANK](https://learn.microsoft.com/en-us/sql/t-sql/functions/rank-transact-sql)** and **[Divide](https://learn.microsoft.com/en-us/sql/t-sql/language-elements/divide-transact-sql)**, Microsoft Learn. The SQL Server references behind section 9.
- **[About MetricFlow](https://docs.getdbt.com/docs/build/about-metricflow)**, dbt. How analytics teams write metric definitions down once, as code, so every dashboard uses the same one; the Analytics Engineering track builds on this idea.
- **Practice:** [DataLemur's RANK vs DENSE_RANK vs ROW_NUMBER tutorial](https://datalemur.com/sql-tutorial/sql-rank-dense_rank-row_number-window-function), the ranking-with-ties question in interview form.

For interview questions on this topic with model answers, see the [study guide](<study-guides/09-thinking-in-metrics.md>).

## Where to go from here

This is the last notebook of the shared SQL foundation. Over nine notebooks you've learned to make SQL return exactly what you asked for, and this one was about the part SQL can't do for you, which is deciding what to ask. From here the curriculum splits by track, into Data Analytics, Analytics Engineering, and Data Engineering, and the [Start Here roadmap](https://aaboutdata.substack.com/p/start-here-from-beginner-to-job-ready) explains what each one covers and which might suit you. Whichever you pick, you'll be writing metrics like these, so learn to ask what the number means before you ask how to compute it.

---

[← 08b - Changing Data, Part 2](<08b-changing-data-part-2.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).